# Hosted vs Client-Side Tools

Every other tool-use notebook in this folder executes tools **on your machine**: the model
returns a `tool_call`, your code runs the function, you hand the result back. You are the
runtime.

There is a second topology. The model calls a tool that runs **on the provider's
infrastructure**, and you receive a reply with the result already folded in. You never see
the execution.

The API difference is a few lines. The operational difference is large, and it is what this
notebook is actually about: **who executes, what you can see, what you can intercept, and
how it fails.**

## Learning objectives

1. Run the same capability both ways and see where the work happens.
2. Name what you give up with a hosted tool, and what you get.
3. Read a tool-call trace and say which topology produced it.
4. Decide between them from requirements rather than habit.

## Where this fits

- Siblings: `02_Tool_Calling_vs_ReAct.ipynb` compares two *loop shapes*. This compares two
  *execution locations* — an orthogonal axis.
- `06_BrowserAgent_Computer_Use_Applied.ipynb` builds a client-side tool over a simulated
  environment. Computer use is the hosted version of that idea.

## On the Responses API

Hosted tools exist **only** on the Responses API — verified in the installed SDK:
`openai.types.responses` defines `FileSearchTool`, `WebSearchTool`, `ComputerUsePreviewTool`
and friends, while `openai.types.chat` has only `Function` and `Custom` tools, both
client-side. You cannot demonstrate a hosted tool without it.

So this notebook uses `responses.create` **as a vehicle, not as a subject**. We are not
teaching the Responses API's statefulness or `previous_response_id` chaining — LangGraph
checkpointing already covers conversation state in this repo. We use it because it is the
only door to the thing being taught.

## Prerequisites

`OPENAI_API_KEY` in the project-root `.env`, on an account with hosted tools enabled. The
hosted calls are billed per tool invocation on top of tokens. A handful of cents.

This notebook uses the raw `openai` client rather than `helpers.get_llm()`, because LangChain
does not surface hosted-tool configuration. `02_Tool_Calling_vs_ReAct.ipynb` in this folder
sets the same precedent of going direct when the subject demands it.

In [ ]:
# ============ SETUP ============
import json
import time

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI()

MODEL = "gpt-4o-mini"        # client-side path
HOSTED_MODEL = "gpt-4o-mini" # hosted path; must be a model your account can use tools with

QUESTION = "What is the current population of Lisbon? Answer in one short sentence."

## 1. Client-side: you are the runtime

The familiar loop. Four steps, and **you own steps 2 and 3**:

1. You declare the tool's schema.
2. The model replies with a `tool_call` — arguments only, no execution.
3. **You** run the function and decide what the result is.
4. You send the result back and the model writes the answer.

The thing to notice is how much control lives in step 3. You can log the arguments, refuse
the call, sanitise the result, cache it, swap the implementation, or fail it deliberately.

In [ ]:
# ============ CLIENT-SIDE TOOL ============
def lookup_population(city: str) -> str:
    """Stands in for a real API call. Deterministic so the run is reproducible."""
    print(f"    [your machine] lookup_population(city={city!r}) executing here")
    return json.dumps({"city": city, "population": "545,796", "source": "local stub"})


client_tools = [{
    "type": "function",
    "function": {
        "name": "lookup_population",
        "description": "Look up the population of a city.",
        "parameters": {
            "type": "object",
            "properties": {"city": {"type": "string"}},
            "required": ["city"],
        },
    },
}]

started = time.perf_counter()
messages = [{"role": "user", "content": QUESTION}]

first = client.chat.completions.create(model=MODEL, messages=messages, tools=client_tools)
call = first.choices[0].message.tool_calls[0]
print(f"  1. model asked for : {call.function.name}({call.function.arguments})")

result = lookup_population(**json.loads(call.function.arguments))
print(f"  2. you returned    : {result}")

messages += [first.choices[0].message,
             {"role": "tool", "tool_call_id": call.id, "content": result}]
second = client.chat.completions.create(model=MODEL, messages=messages, tools=client_tools)

client_secs = time.perf_counter() - started
print(f"\n  answer: {second.choices[0].message.content}")
print(f"  round trips: 2 · {client_secs:.2f}s")

### What that trace shows

Two round trips, and a visible gap between them where **your code ran**. That gap is the
whole difference. Everything you would want to do operationally — audit the arguments,
rate-limit, inject a cached answer, redact the result before the model sees it — happens
there, because you are holding the execution.

## 2. Hosted: the provider is the runtime

Now the same capability with a tool that runs on OpenAI's side. Note what disappears from
the code: there is no function to write, no loop, no second call. You declare a tool *type*
and get an answer.

In [ ]:
# ============ HOSTED TOOL ============
started = time.perf_counter()

hosted = client.responses.create(
    model=HOSTED_MODEL,
    input=QUESTION,
    tools=[{"type": "web_search"}],   # runs on OpenAI's infrastructure, not yours
)

hosted_secs = time.perf_counter() - started
print(f"  answer: {hosted.output_text}")
print(f"  round trips: 1 · {hosted_secs:.2f}s")
print("\n  note: no 'executing here' line printed — nothing ran on your machine")

In [ ]:
# ============ WHAT CAME BACK ============
# The response carries the tool activity as output items. This is the ONLY window you get
# into an execution you did not perform.
for item in hosted.output:
    kind = getattr(item, "type", "?")
    print(f"  output item: {kind}")
    if kind != "message":
        # hosted tool items expose status and sometimes the query, but never the raw result
        for attr in ("status", "action", "queries"):
            if hasattr(item, attr):
                print(f"      {attr}: {getattr(item, attr)}")

### What that trace shows — and what it does not

One round trip. No function of yours ran. The output items tell you a tool was *used*, and
roughly what it did, but you did not choose the query, cannot see the raw result, and had no
opportunity to intervene between retrieval and generation.

That is the trade in one sentence: **you exchanged control for a tool you did not have to
build or operate.**

## 3. The comparison that matters

Not "which is better" — they fail differently, and the right answer follows from
requirements.

In [ ]:
# ============ SIDE BY SIDE ============
rows = [
    ("who executes",          "your process",                  "provider infrastructure"),
    ("round trips",           "2+ (one per tool call)",        "1"),
    ("code you maintain",     "the function, and its deps",    "none"),
    ("see the arguments",     "yes, before execution",         "sometimes, after the fact"),
    ("see the raw result",    "yes",                           "depends on the tool"),
    ("intercept / veto",      "yes — it is your code",         "no"),
    ("cache or stub it",      "trivially",                     "not at all"),
    ("deterministic in tests","yes, stub the function",        "no, it hits the live world"),
    ("data leaves your env",  "only what you send",            "the tool acts on their side"),
    ("failure surface",       "your exception, your retry",    "provider status, opaque"),
    ("cost model",            "tokens + your infra",           "tokens + per-tool-call fee"),
]
print(f"{'':<24}{'client-side':<34}{'hosted'}")
print("-" * 92)
for label, a, b in rows:
    print(f"{label:<24}{a:<34}{b}")

print(f"\nthis run: client-side {client_secs:.2f}s (2 trips) · hosted {hosted_secs:.2f}s (1 trip)")

## 4. The part that actually bites: failure

A client-side tool fails in your process. You get an exception, at a line number, and you
decide what happens next — retry, fall back, degrade, surface an error.

A hosted tool fails on the other side of an API boundary. You get a status on an output
item. You cannot retry *just the tool*, because you did not invoke it; you can only re-run
the whole request and hope. And a hosted tool that returns something subtly wrong is worse
than one that errors — you have no raw result to inspect, so the first sign of trouble is a
bad final answer with no trace explaining it.

That is the argument for keeping anything **consequential** client-side: not that hosted
tools are unreliable, but that when they misbehave you have no seam to debug at.

In [ ]:
# ============ THE SEAM YOU ONLY HAVE CLIENT-SIDE ============
# Same tool, now refusing a call. This is impossible with a hosted tool: by the time you
# see anything, the tool has already run.
def guarded_lookup(city: str) -> str:
    if city.strip().lower() not in {"lisbon", "porto"}:
        print(f"    [your machine] REFUSED {city!r} — outside allowed scope")
        return json.dumps({"error": "city not in allowed scope"})
    return lookup_population(city)


messages = [{"role": "user", "content": "What is the population of Pyongyang?"}]
first = client.chat.completions.create(model=MODEL, messages=messages, tools=client_tools)
call = first.choices[0].message.tool_calls[0]
refused = guarded_lookup(**json.loads(call.function.arguments))

messages += [first.choices[0].message,
             {"role": "tool", "tool_call_id": call.id, "content": refused}]
final = client.chat.completions.create(model=MODEL, messages=messages, tools=client_tools)
print(f"\n  model's reply after refusal: {final.choices[0].message.content}")

### Why that cell is the point of the notebook

The guard is four lines, and it exists **only because execution is yours**. Policy
enforcement, allow-lists, tenant scoping, PII redaction, spend caps — all of it lives in that
seam. Choosing a hosted tool means choosing not to have it.

Which is often fine. Web search over public data has little to enforce. The judgement is
about the specific tool, not the topology.

## 5. Choosing

| Reach for **hosted** when | Reach for **client-side** when |
|---|---|
| The capability is a commodity you would otherwise rebuild — web search, a Python sandbox | The tool touches your data, your systems, or your customers |
| Nothing needs enforcing between the call and the result | You need an allow-list, tenant scoping, redaction, or a spend cap |
| You want it working today and operating it is not the point | You need deterministic tests — stub the function and the whole path is reproducible |
| Opaque failure is acceptable | You need to debug it in six months from a trace |

A useful default: **hosted for commodity capability, client-side for anything
consequential** — and be honest that "consequential" usually means "touches a customer".

## Key takeaways

1. **Hosted vs client-side is an execution-location choice, not an API detail.** The code
   difference is small; the operational difference is not.
2. **Client-side gives you a seam.** Between "model asked" and "model sees the result" you can
   log, refuse, cache, redact or substitute. Hosted removes that seam entirely.
3. **How much of a hosted tool you can see depends on the tool.** `web_search` returns a
   conclusion and little else. `code_interpreter` returns the *source it executed* —
   see `09_Hosted_Code_Execution.ipynb`, which works through that case and the container
   controls that come with it. Do not generalise from the tool in this notebook.
4. **Failures are the real difference.** Client-side: your exception, your retry. Hosted: a
   status across an API boundary, no way to retry the tool alone, nothing to inspect when the
   answer is merely wrong rather than erroring.
5. **Hosted tools cost per invocation** on top of tokens, so the cheaper-looking single round
   trip is not automatically cheaper.
6. **Hosted tools live only on the Responses API.** Chat Completions has function tools only —
   an SDK-level fact, not a documentation quirk.

### Next

- `06_BrowserAgent_Computer_Use_Applied.ipynb` — the client-side version of computer use,
  over a simulated environment.
- `03_Advanced/09_Agent_Protocols/MCP/` — a third topology: the tool runs on a server *you*
  control, reached over a protocol. It gives back much of the seam hosted tools remove.